In [1]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.amp import autocast, GradScaler

from PIL import Image
from PIL import ImageFile

import pandas
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import os

from tqdm import tqdm
import copy

In [ ]:
class FilteredImageFolder(ImageFolder):
    def __init__(self, root, excluded_classes=None, **kwargs):
        self.excluded_classes = set(excluded_classes) if excluded_classes else set()
        super().__init__(root, **kwargs)

    def find_classes(self, directory):
        # Let the parent find all classes first
        classes, class_to_idx = super().find_classes(directory)

        # Filter out the unwanted classes
        if self.excluded_classes:
            classes = [c for c in classes if c not in self.excluded_classes]

        # Re-build the dictionary to ensure indices are contiguous (0, 1, 2...)
        # If we didn't do this, removing the middle folder might result in indices [0, 2]
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

        return classes, class_to_idx

In [3]:
class ImageRecognitionDataset(Dataset):
    def __init__(self,data_dir,transform):
        self.data = FilteredImageFolder(data_dir,transform=transform,excluded_classes=["RealArt"])
    def __len__(self):
        return len(self.data)
    def __getitem__(self,index):
        return self.data[index]
    @property
    def classes(self):
        return self.data.classes

lr = 0.0005
n_epochs = 10
mean = [0.4932, 0.4570, 0.4053] #ImageNet: [0.485, 0.456, 0.406] Calculated: [0.4932, 0.4570, 0.4053]
std = [0.3003, 0.2854, 0.2941] #ImageNet: [0.229, 0.224, 0.225] Calculated: [0.3003, 0.2854, 0.2941]

transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize(256),
    transforms.RandomCrop((256,256)), #Not using randomresizedcrop since images are resized to 256 by 256.
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.03),
    #transforms.GaussianBlur(kernel_size=(3, 3),sigma=(0.2,0.2)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean = mean, std = std)
])
transform_test = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize(256),
    transforms.RandomCrop((256,256)),
    transforms.ToTensor(),
    transforms.Normalize(mean = mean,std = std)
])

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

In [ ]:
train_set = ImageRecognitionDataset(data_dir="/kaggle/input/diffusion-image-detection/Set/TestSet",transform=transform)
test_set= ImageRecognitionDataset(data_dir="/kaggle/input/training-set/train/train",transform=transform_test)
val_set = ImageRecognitionDataset(data_dir="/kaggle/input/tiny-genimage/imagenet_ai_0508_adm/train",transform=transform_test)

print(train_set.classes)

train_loader = DataLoader(train_set,batch_size=32,shuffle=True,num_workers=4)
test_loader = DataLoader(test_set,batch_size=32,shuffle=False,num_workers=4)
val_loader = DataLoader(val_set,batch_size=32,shuffle=False,num_workers=4)

In [ ]:
#Can ignore
def imshow(img):
    img = img.clone()  # avoid modifying original
    img = torch.clamp(img, 0, 1)  # ensure valid range
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# one batch of random training images
dataiter = iter(train_loader)
images, labels = next(dataiter)
img_grid = torchvision.utils.make_grid(images[0:25], nrow=5)
imshow(img_grid)
imshow(images[0])

In [ ]:
#Customising model to train
device = torch.device('cuda')

model = torchvision.models.efficientnet_b0(weights=torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1)
# model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features=1280, out_features=2)
)
#print(model) #View model architecture
model.train()
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr,weight_decay = 0.000025)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max = n_epochs, eta_min=5e-6)

for i in range(4): # 0 (Stem) up to 3 (Stage 3) For efficientnetb0
    for param in model.features[i].parameters():
        param.requires_grad = False

# for name, param in model.named_parameters(): #For ResNet50
#     if "layer4" in name or "layer3" in name or "fc" in name:
#         param.requires_grad = True
#     else:
#         param.requires_grad = False

In [ ]:
#Training loop
scaler = GradScaler()

patience = 5
best_val_loss = float('inf')
train_loss_arr = []
val_loss_arr = []
epochs_no_improv = 0
epochs_cycled = 0
best_model_weights = copy.deepcopy(model.state_dict())
best_model_epoch = 0

n_total_steps = len(train_loader)
for epoch in tqdm(range(n_epochs)):
    epochs_cycled += 1
    print(f"Training epoch {epoch}.")
    model.train()
    running_loss = 0
    for i, (images,labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}",position=0,leave=True)):
        optimizer.zero_grad()
        images = images.to(device)
        labels = labels.to(device)
        with torch.autocast(device_type="cuda",dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs,labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
    scheduler.step()
    print(f'[{epoch + 1}]\nTraining loss: {running_loss / n_total_steps:.3f}')
    train_loss_arr.append(float(f"{running_loss / n_total_steps:.3f}"))
    
    #Evaluation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        n_correct = 0
        n_samples = len(test_loader.dataset)
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            n_correct += (predicted == labels).sum().item()
        acc = 100.0 * n_correct / n_samples
        print(f'Accuracy of the model: {acc} %')
        print(f"Validation loss: {val_loss/len(test_loader):.3f}")
        val_loss_arr.append(float(f"{val_loss/len(test_loader):.3f}"))
    if val_loss/len(test_loader) < best_val_loss:
        best_val_loss = val_loss/len(test_loader)
        best_model_weights = copy.deepcopy(model.state_dict())
        best_model_epoch = epoch+1
        epochs_no_improv = 0
    else:
        epochs_no_improv += 1
    if epochs_no_improv > patience:
        print(f"No improvement after {patience} epochs, stopping training.")
        break
print('Finished Training')
print(f"Best model epoch: {best_model_epoch}")



In [ ]:
#Saving the model
save_folder = os.listdir('/kaggle/input/centad_cnn-v1.1/pytorch/default/13') #Update this please.
folder_length = len(save_folder) + 1

PATH = f'/kaggle/working/centad_v1.{folder_length}.pth'
model.load_state_dict(best_model_weights)
torch.save(model.state_dict(), PATH)

In [ ]:
#Display model loss throughout training
epochs = range(1, epochs_cycled+1)

#plt.figure(figsize=(10, 6))
fig,ax = plt.subplots()
plt.plot(epochs, train_loss_arr, label='Training Loss', color='blue')
plt.plot(epochs, val_loss_arr, label='Validation Loss', color='red')


plt.title('Training and Validation Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.legend() # Display the labels for each curve
plt.grid(True) # Add grid for readability
plt.show()

In [ ]:
val_set = ImageRecognitionDataset(data_dir="Insert dataset path here",transform=transform_test)
val_loader = DataLoader(val_set,batch_size=32,shuffle=False,num_workers=4)

device = torch.device('cuda')
#Change this based on model version
PATH = '~/centad_v1.10.pth'

loaded_model = torchvision.models.efficientnet_b0()
loaded_model.classifier = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features=1280, out_features=2)
)

loaded_model.load_state_dict(torch.load(PATH)) #GPU
# loaded_model.load_state_dict(torch.load(PATH,map_location=torch.device('cpu'))) #CPU
loaded_model.eval()
loaded_model.to(device)
criterion = nn.CrossEntropyLoss()
val_loss = 0
with torch.no_grad():
    n_correct = 0
    n_samples = len(val_loader.dataset)
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = loaded_model(images)
        loss = criterion(outputs,labels)
        val_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        n_correct += (predicted == labels).sum().item()
    acc = 100.0 * n_correct / n_samples
    print(f'Accuracy of the model: {acc} %\nModel loss: {val_loss/len(val_loader):.3f}')